# Generating Iot_sensor DataSet:-2

In [1]:

from datetime import datetime, timedelta
import numpy as np
import pandas as pd

# 1. Load the raw shipments file
shipments_df = pd.read_csv(r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\AUTOSIN\vaccine_shipments_raw.xls")

# Target 500 shipments x 100 pings = 50,000 rows
shipments_subset = (
    shipments_df.drop_duplicates(subset=["shipment_id"]).head(500).copy()
)

# Geographic coordinate mapping for Indian hubs
CITY_COORDS = {
    "mumbai": (19.0760, 72.8777),
    "kochi": (9.9312, 76.2673),
    "bengaluru": (12.9716, 77.5946),
    "thiruvananthapuram": (8.5241, 76.9366),
    "pune": (18.5204, 73.8567),
    "hyderabad": (17.3850, 78.4867),
    "delhi": (28.7041, 77.1025),
    "chennai": (13.0827, 80.2707),
    "kolkata": (22.5726, 88.3639),
    "ahmedabad": (23.0225, 72.5714),
}

records = []
reading_id = 1
np.random.seed(42)

print("Generating 50,000 IoT sensor records from shipments dataset...")

for _, row in shipments_subset.iterrows():
  shp_id = str(row["shipment_id"])
  cnt_id = str(row["container_id"])
  min_temp = float(row["min_temperature"]) if pd.notna(row["min_temperature"]) else 2.0
  max_temp = float(row["max_temperature"]) if pd.notna(row["max_temperature"]) else 8.0

  # Safe date parsing
  arr_dt = pd.to_datetime(row["expected_arrival"], errors="coerce")
  dep_dt = pd.to_datetime(row["departure_time"], errors="coerce")

  if pd.isna(arr_dt):
    arr_dt = datetime(2026, 8, 15, 12, 0, 0)
  if pd.isna(dep_dt) or dep_dt >= arr_dt:
    dep_dt = arr_dt - timedelta(hours=24)

  # Coordinate interpolation
  src_key = str(row["source"]).strip().lower()
  dst_key = str(row["destination"]).strip().lower()

  src_lat, src_lon = CITY_COORDS.get(src_key, (17.3850, 78.4867))
  dst_lat, dst_lon = CITY_COORDS.get(dst_key, (19.0760, 72.8777))

  total_seconds = max((arr_dt - dep_dt).total_seconds(), 3600)
  step_seconds = total_seconds / 100.0

  # ~8% chance of refrigeration compressor failure
  has_failure = np.random.rand() < 0.08
  temp_target = (min_temp + max_temp) / 2.0
  cur_temp = temp_target

  for ping in range(100):
    t_stamp = dep_dt + timedelta(seconds=ping * step_seconds)
    progress = (ping + 1) / 100.0

    # Moving coordinates
    cur_lat = round(
        src_lat + (dst_lat - src_lat) * progress + np.random.uniform(-0.01, 0.01),
        4,
    )
    cur_lon = round(
        src_lon + (dst_lon - src_lon) * progress + np.random.uniform(-0.01, 0.01),
        4,
    )

    # Temperature failure drift vs normal fluctuation
    if has_failure and ping > 35:
      cur_temp += np.random.uniform(0.12, 0.35)
    else:
      cur_temp = temp_target + np.random.uniform(-0.4, 0.4)

    humidity = round(float(np.random.uniform(40.0, 65.0)), 2)
    vibration = round(
        float(
            np.random.uniform(0.02, 0.18)
            if np.random.rand() > 0.05
            else np.random.uniform(0.40, 0.85)
        ),
        3,
    )

    records.append({
        "reading_id": f"READ{reading_id:07d}",
        "shipment_id": shp_id,
        "container_id": cnt_id,
        "timestamp": t_stamp.strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": round(cur_temp, 2),
        "humidity": humidity,
        "vibration": vibration,
        "latitude": cur_lat,
        "longitude": cur_lon,
    })
    reading_id += 1

# Export to CSV
iot_df = pd.DataFrame(records)
iot_df.to_csv("iot_sensor_readings.csv", index=False)

print(f"Success! Created 'iot_sensor_readings.csv' with {len(iot_df):,} rows.")
iot_df.head()

Generating 50,000 IoT sensor records from shipments dataset...


C:\Users\VISHAL\AppData\Local\Temp\ipykernel_22940\3526389327.py:41: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dep_dt = pd.to_datetime(row["departure_time"], errors="coerce")


Success! Created 'iot_sensor_readings.csv' with 50,000 rows.


,reading_id,shipment_id,container_id,timestamp,temperature,humidity,vibration,latitude,longitude
0,READ0000001,SHP0001,CNT1001,2026-08-08 13:15:00,5.08,43.90,0.029,10.0317,76.2380
1,READ0000002,SHP0001,CNT1001,2026-08-08 13:30:00,5.17,40.51,0.153,10.1214,76.2015
2,READ0000003,SHP0001,CNT1001,2026-08-08 13:45:00,4.75,47.61,0.089,10.1998,76.1592
3,READ0000004,SHP0001,CNT1001,2026-08-08 14:00:00,4.71,47.30,0.093,10.2928,76.1340
4,READ0000005,SHP0001,CNT1001,2026-08-08 14:15:00,5.01,54.81,0.673,10.3941,76.0918


In [2]:
iot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   reading_id    50000 non-null  object 
 1   shipment_id   50000 non-null  object 
 2   container_id  50000 non-null  object 
 3   timestamp     50000 non-null  object 
 4   temperature   50000 non-null  float64
 5   humidity      50000 non-null  float64
 6   vibration     50000 non-null  float64
 7   latitude      50000 non-null  float64
 8   longitude     50000 non-null  float64
dtypes: float64(5), object(4)
memory usage: 3.4+ MB


In [3]:
iot_df.to_csv("iot_sensor_readings.csv", index=False)

# Generate Route Dataset:4

In [8]:
import itertools
import numpy as np
import pandas as pd

# 1. Load actual origins and destinations from your shipments
shipments_df = pd.read_csv(r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\AUTOSIN\vaccine_shipments_raw.xls")
shipment_routes = (
    shipments_df[["source", "destination"]].dropna().drop_duplicates()
)

# Standardized coordinates for all operational hubs

In [9]:
CITY_COORDS = {
    "Mumbai": (19.0760, 72.8777),
    "Delhi": (28.7041, 77.1025),
    "Bengaluru": (12.9716, 77.5946),
    "Hyderabad": (17.3850, 78.4867),
    "Ahmedabad": (23.0225, 72.5714),
    "Chennai": (13.0827, 80.2707),
    "Kolkata": (22.5726, 88.3639),
    "Pune": (18.5204, 73.8567),
    "Kochi": (9.9312, 76.2673),
    "Thiruvananthapuram": (8.5241, 76.9366),
    "Jaipur": (26.9124, 75.7873),
    "Lucknow": (26.8467, 80.9462),
    "Nagpur": (21.1458, 79.0882),
    "Patna": (25.5941, 85.1376),
    "Bhubaneswar": (20.2961, 85.8245),
    "Chandigarh": (30.7333, 76.7794),
    "Indore": (22.7196, 75.8577),
    "Bhopal": (23.2599, 77.4126),
    "Visakhapatnam": (17.6868, 83.2185),
    "Surat": (21.1702, 72.8311),
    "Vadodara": (22.3072, 73.1812),
    "Coimbatore": (11.0168, 76.9558),
    "Madurai": (9.9252, 78.1198),
    "Guwahati": (26.1445, 91.7362),
    "Varanasi": (25.3176, 82.9739),
}


def haversine_road_dist(lat1, lon1, lat2, lon2):
  R = 6371.0
  phi1, phi2 = np.radians(lat1), np.radians(lat2)
  dphi = np.radians(lat2 - lat1)
  dlambda = np.radians(lon2 - lon1)
  a = (
      np.sin(dphi / 2.0) ** 2
      + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
  )
  c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
  return round(R * c * 1.25, 1)


route_list = []
seen_pairs = set()

## A. First guarantee all real shipment routes (91 unique pairs)

In [10]:
for _, row in shipment_routes.iterrows():
  src = str(row["source"]).strip().title()
  dst = str(row["destination"]).strip().title()
  if (
      src != dst
      and (src, dst) not in seen_pairs
      and src in CITY_COORDS
      and dst in CITY_COORDS
  ):
    seen_pairs.add((src, dst))
    lat1, lon1 = CITY_COORDS[src]
    lat2, lon2 = CITY_COORDS[dst]
    dist = haversine_road_dist(lat1, lon1, lat2, lon2)
    time_hrs = round(dist / 48.0, 1)  # Avg 48 km/h reefer speed
    route_list.append({
        "source": src,
        "destination": dst,
        "distance_km": dist,
        "estimated_travel_time_hours": time_hrs,
    })

## B. Fill remaining slots up to 200 with secondary rerouting hubs

In [11]:
all_possible_pairs = list(itertools.permutations(list(CITY_COORDS.keys()), 2))
np.random.seed(42)
np.random.shuffle(all_possible_pairs)

for src, dst in all_possible_pairs:
  if len(route_list) >= 200:
    break
  if (src, dst) not in seen_pairs:
    seen_pairs.add((src, dst))
    lat1, lon1 = CITY_COORDS[src]
    lat2, lon2 = CITY_COORDS[dst]
    dist = haversine_road_dist(lat1, lon1, lat2, lon2)
    time_hrs = round(dist / 48.0, 1)
    route_list.append({
        "source": src,
        "destination": dst,
        "distance_km": dist,
        "estimated_travel_time_hours": time_hrs,
    })

# Add primary key route_id

In [12]:

for idx, item in enumerate(route_list, 1):
  item["route_id"] = f"RT{idx:04d}"

# Reorder columns and export
df_useful_routes = pd.DataFrame(route_list)[
    [
        "route_id",
        "source",
        "destination",
        "distance_km",
        "estimated_travel_time_hours",
    ]
]
df_useful_routes.to_csv("routes_raw.csv", index=False)

print(
    f"Successfully generated {len(df_useful_routes)} fully mapped routes in 'routes_raw.csv'."
)
df_useful_routes.head()

Successfully generated 200 fully mapped routes in 'routes_raw.csv'.


,route_id,source,destination,distance_km,estimated_travel_time_hours
0,RT0001,Kochi,Mumbai,1350.2,28.1
1,RT0002,Bengaluru,Delhi,2187.6,45.6
2,RT0003,Thiruvananthapuram,Kochi,216.1,4.5
3,RT0004,Mumbai,Kochi,1350.2,28.1
4,RT0005,Hyderabad,Pune,632.2,13.2
